# Drifting ICNN

In [1]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from typing import Tuple, List

In [2]:
def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True)

### Hyperparameters

In [10]:
METHODS = ["kernel", "ot_direct", "npf"]
NUM_ITERS = 3000
BATCH_SIZE = 128
LATENT_DIM = 64
NOISE_DIM = 64
LR_GEN = 2e-4
LR_VAE = 1e-3
INNER_STEPS = 15
VAE_WARMUP_STEPS = 400
SEED = 42
seed_everything(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MNIST_ROOT = "./data"
SAVE_PATH = "results/mnist/mnist_drifting_results.png"

# NPF (Vesseron & Cuturi, 2024) ICNN parameterization replaces the plain
# ICNN drift field. Hidden widths kept identical to the previous baseline
# so capacity is comparable.
NPF_HIDDEN_DIMS = [128, 128, 128, 128]
NPF_OUTER_RANK = 4
NPF_INNER_RANK = 1
# softplus has f''>0 everywhere, so the cascade can carry curvature from
# the first inner step (ELU's u≥0 has f''=0 at the operating point).
NPF_ACTIVATION = "softplus"
NPF_ELU_ALPHA = 1.0
NPF_SOFTPLUS_BETA = 10.0
# init_eps>0 makes b_linears.weight live-at-init (NOT zero); breaks the
# rank-1 symmetry in ∂L/∂(cascade params) while preserving T(x)≈x.
NPF_INIT_EPS = 1e-2

# Adam hyperparameters for the NPF inner loop.
NPF_INNER_LR = 3e-3
NPF_ADAM_BETAS = (0.9, 0.999)
NPF_ADAM_EPS = 1e-8
NPF_WEIGHT_DECAY = 0.0
NPF_GRAD_CLIP = 5.0

SINKHORN_REG = 1.3


### 1. The VAE (Encoder/Decoder)
We need this to map Images <-> Latent Space. 
The Drifting Model operates ONLY in the latent space.

In [11]:
class MNISTVAE(nn.Module):
    def __init__(self, latent_dim: int = 64):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=4, stride=2, padding=1),  # 28 -> 14
            nn.SiLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1), # 14 -> 7
            nn.SiLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),# 7 -> 4
            nn.SiLU(),
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.SiLU(),
        )
        self.fc_mu = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)

        self.decoder_fc = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.SiLU(),
            nn.Linear(256, 128 * 4 * 4),
            nn.SiLU(),
        )
        self.decoder = nn.Sequential(
            nn.Unflatten(1, (128, 4, 4)),
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=0), # 4 -> 7
            nn.SiLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),                     # 7 -> 14
            nn.SiLU(),
            nn.ConvTranspose2d(32, 1, kernel_size=4, stride=2, padding=1),                      # 14 -> 28
            nn.Sigmoid(),
        )

    def encode(self, x: torch.Tensor):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z: torch.Tensor):
        h = self.decoder_fc(z)
        return self.decoder(h)

    def forward(self, x: torch.Tensor):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar


class LatentGenerator(nn.Module):
    def __init__(self, noise_dim: int = 64, latent_dim: int = 64, hidden_dim: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, latent_dim),
        )
        self._init_output_scale(target_std=1.0)

    def _init_output_scale(self, target_std: float):
        with torch.no_grad():
            test_input = torch.randn(2000, self.net[0].in_features)
            test_output = self.net(test_input)
            current_std = test_output.std().item()
            if current_std > 1e-6:
                scale_factor = target_std / current_std
                self.net[-1].weight.mul_(scale_factor)
                self.net[-1].bias.mul_(scale_factor)

    def forward(self, eps: torch.Tensor):
        return self.net(eps)

### 2. NPF ICNN & Sinkhorn (NPF parameterization of $\psi_\omega$)
The convex potential is now an NPF (Vesseron & Cuturi, 2024) input convex network: PSD outer base + per-layer convex quadratic injections + non-negative cascade. Both the principled LogNormal init and the identity init are applied jointly — they are complementary, not alternatives. All NPF code lives in [`npf_drift.py`](npf_drift.py) and is shared by every notebook in this folder.

In [12]:
# Shared NPF module — same architecture as the LR_CIFAR10 NPF ICNN, ported
# here for the MNIST latent-space drifting experiment.
#
# `barycentric_target_simple` is the same direct-domain Sinkhorn target the
# previous inline `sinkhorn` function produced; we use it both as the
# direct OT baseline (compute_V_ot_direct) and as the Sinkhorn target
# function fed to NPFDriftField.
from npf_drift import (
    NPFInputConvexPotential,
    NPFDriftField,
    barycentric_target_simple,
    sinkhorn_simple,
    count_parameters,
)


def make_npf_sinkhorn_target_fn(reg: float):
    """Closure over reg for NPFDriftField.sinkhorn_target_fn."""
    return lambda x, y: barycentric_target_simple(x, y, reg=reg, num_iters=100)


In [13]:
# NPFDriftField factory for this notebook. The previous inline
# ICNN + ICNNDriftField classes are replaced by NPFInputConvexPotential
# + NPFDriftField from npf_drift.py. All of the joint init logic
# (principled LogNormal + identity init with live-at-init b_linears and
# b_out weights) is handled inside the module — see
# `NPFInputConvexPotential.init_as_identity`'s docstring for the why.


def make_npf_drift(dim: int = LATENT_DIM,
                   hidden_dims=None,
                   inner_steps: int = INNER_STEPS,
                   sinkhorn_reg: float = SINKHORN_REG,
                   inner_lr: float = NPF_INNER_LR,
                   init_eps: float = NPF_INIT_EPS,
                   outer_rank: int = NPF_OUTER_RANK,
                   inner_rank: int = NPF_INNER_RANK,
                   activation: str = NPF_ACTIVATION,
                   softplus_beta: float = NPF_SOFTPLUS_BETA,
                   grad_clip: float = NPF_GRAD_CLIP) -> NPFDriftField:
    """Construct an identity-init NPFDriftField with this notebook's defaults.

    The Sinkhorn target function uses the same direct-domain solver and
    `reg` value the previous inline ICNNDriftField used (see
    `barycentric_target_simple` in npf_drift), so behaviour is comparable
    apart from the parameterization of ψ.
    """
    if hidden_dims is None:
        hidden_dims = NPF_HIDDEN_DIMS
    return NPFDriftField(
        dim=dim,
        hidden_dims=hidden_dims,
        outer_rank=outer_rank,
        inner_rank=inner_rank,
        activation=activation,
        elu_alpha=NPF_ELU_ALPHA,
        softplus_beta=softplus_beta,
        init_eps=init_eps,
        inner_steps=inner_steps,
        inner_lr=inner_lr,
        adam_betas=NPF_ADAM_BETAS,
        adam_eps=NPF_ADAM_EPS,
        weight_decay=NPF_WEIGHT_DECAY,
        grad_clip=grad_clip,
        sinkhorn_reg=sinkhorn_reg,
        sinkhorn_iters=100,
        sinkhorn_target_fn=make_npf_sinkhorn_target_fn(sinkhorn_reg),
        init_mode="identity",
    )


### 3. Baselines (Kernel & OT Direct) - Adapted for Latent Space

In [14]:
def compute_V_kernel(x_gen: torch.Tensor, y_pos: torch.Tensor,
                     tau_list: Tuple[float, ...] = (0.02, 0.05, 0.2)):
    """
    Compute the drifting field V(x) following the original Drifting Model (Deng et al.)
    Algorithm 2 / Eq. 11 of the paper.

    Returns V (detached), such that the training loss is:
        target = sg(x_gen + V)
        loss = ||x_gen - target||²

    The loss value equals ||V||², which DECREASES as q → p.
    """
    N, D = x_gen.shape
    M = y_pos.shape[0]
    x = x_gen.detach()
    y = y_pos.detach()

    # Targets: [negatives (=generated), positives (=data)]
    targets = torch.cat([x, y], dim=0)  # [N+M, D]

    # ── Pairwise ℓ₂ distances between generated samples and all targets ──
    dist = torch.cdist(x, targets)  # [N, N+M]

    # ── Distance normalization (Appendix A.6) ──
    # Normalize so mean pairwise distance ≈ √D
    scale = dist.mean().clamp(min=1e-6)
    dist_normed = dist * (np.sqrt(D) / scale)

    # ── Self-masking, to prevent trivial zero-distance matches between generated samples and themselves ──
    diag_mask = torch.zeros(N, N + M, device=x.device)
    diag_mask[:, :N] = torch.eye(N, device=x.device)
    dist_normed = dist_normed + diag_mask * 100.0

    # ── Force accumulation (NO per-temperature normalization) ──
    V = torch.zeros_like(x)

    for tau in tau_list:
        # Sinkhorn-like affinities: a_ij = exp(-d_ij / tau)
        # with double softmax normalization (over rows and columns) to ensure better gradients and prevent collapse
        # The tau (temperature) controls the sharpness of the affinities: smaller tau → sharper affinities, more emphasis on closer targets
        logits = -dist_normed / tau

        # Double softmax (paper's Alg. 2: softmax over y, then over x)
        aff_row = torch.softmax(logits, dim=-1)
        aff_col = torch.softmax(logits, dim=-2)
        affinity = torch.sqrt((aff_row * aff_col).clamp(min=1e-6))

        aff_neg = affinity[:, :N]                   # negative affinities (to generated samples)
        aff_pos = affinity[:, N:]                   # positive affinities (to data samples)
        sum_pos = aff_pos.sum(-1, keepdim=True)
        sum_neg = aff_neg.sum(-1, keepdim=True)

        # Drifting coefficients: attract by positives, repel by negatives
        r_coeff_neg = -aff_neg * sum_pos
        r_coeff_pos = aff_pos * sum_neg
        R_coeff = torch.cat([r_coeff_neg, r_coeff_pos], dim=1)  # [N, N+M]

        # Force = weighted combination of aim directions (target - x) vectors
        force_R = R_coeff @ targets - R_coeff.sum(-1, keepdim=True) * x
        V = V + force_R  # Raw force, NO normalization

    return V.detach()


def compute_V_ot_direct(x_gen: torch.Tensor, y_pos: torch.Tensor,
                        reg: float = SINKHORN_REG):
    """Direct OT displacement V(x) = T_Sinkhorn(x) - x.

    Uses the same direct-domain barycentric Sinkhorn target as NPF (via
    `barycentric_target_simple` from npf_drift), so any difference between
    the two methods comes from the NPF amortization rather than from
    different OT hyperparameters.
    """
    x = x_gen.detach()
    y = y_pos.detach()
    y_target = barycentric_target_simple(x, y, reg=reg, num_iters=100)
    return (y_target - x).detach()


### 4. Training Loop for MNIST

In [15]:
def _next_batch(data_iter, loader):
    try:
        x_real, _ = next(data_iter)
    except StopIteration:
        data_iter = iter(loader)
        x_real, _ = next(data_iter)
    return x_real, data_iter


def train_mnist(method='kernel', num_iters=NUM_ITERS, batch_size=BATCH_SIZE,
                lr_gen=LR_GEN, inner_steps=INNER_STEPS, device=DEVICE):
    device = torch.device(device)

    # 1) Data
    transform = transforms.Compose([transforms.ToTensor()])
    dataset = datasets.MNIST(root=MNIST_ROOT, train=True, download=True, transform=transform)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True, num_workers=2, pin_memory=torch.cuda.is_available())
    data_iter = iter(loader)

    # 2) VAE warmup for a stable latent manifold
    vae = MNISTVAE(latent_dim=LATENT_DIM).to(device)
    vae_optim = optim.Adam(vae.parameters(), lr=LR_VAE)
    print(f"[{method}] Warming up VAE for {VAE_WARMUP_STEPS} steps...")
    vae.train()
    for _ in range(VAE_WARMUP_STEPS):
        x_real, data_iter = _next_batch(data_iter, loader)
        x_real = x_real.to(device, non_blocking=True)

        recon, mu, logvar = vae(x_real)
        rec_loss = F.binary_cross_entropy(recon, x_real)
        kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
        vae_loss = rec_loss + 1e-3 * kl_loss

        vae_optim.zero_grad()
        vae_loss.backward()
        vae_optim.step()
    print(f"[{method}] VAE warmup done.")

    # 3) Latent generator + drift model
    gen = LatentGenerator(noise_dim=NOISE_DIM, latent_dim=LATENT_DIM).to(device)
    gen_optim = optim.Adam(gen.parameters(), lr=lr_gen)

    npf_drift = None
    if method == 'npf':
        npf_drift = make_npf_drift(
            dim=LATENT_DIM,
            hidden_dims=NPF_HIDDEN_DIMS,
            inner_steps=inner_steps,
            sinkhorn_reg=SINKHORN_REG,
        ).to(device)
    elif method not in {'kernel', 'ot_direct'}:
        raise ValueError(f"Unknown method: {method}")

    losses, v_norms, snapshots = [], [], []
    vae.eval()

    snapshot_period = max(1, num_iters // 8)

    for it in range(num_iters):
        x_real_img, data_iter = _next_batch(data_iter, loader)
        x_real_img = x_real_img.to(device, non_blocking=True)

        with torch.no_grad():
            mu, _ = vae.encode(x_real_img)
            y_pos = mu  # Deterministic latent targets reduce OT noise.


        eps = torch.randn(batch_size, NOISE_DIM, device=device)
        x_gen = gen(eps)

        if method == 'kernel':
            V = compute_V_kernel(x_gen, y_pos, tau_list=(0.5, 1.0, 2.0))
        elif method == 'ot_direct':
            V = compute_V_ot_direct(x_gen, y_pos, reg=SINKHORN_REG)
        else:
            V = npf_drift.compute_V(x_gen, y_pos)

        target_pts = (x_gen.detach() + V).detach()
        loss_drift = ((x_gen - target_pts) ** 2).mean()
        v_norm = (V ** 2).mean().item()

        # Keep generated latents near N(0, I) so they stay decodable.
        reg = 0.5 * (x_gen.pow(2).mean() - torch.log(x_gen.pow(2).mean() + 1e-6))
        loss = loss_drift + 0.01 * reg

        gen_optim.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(gen.parameters(), 1.0)
        gen_optim.step()

        losses.append(loss.item())
        v_norms.append(v_norm)

        if it % snapshot_period == 0 or it == num_iters - 1:
            with torch.no_grad():
                z_eval = gen(torch.randn(64, NOISE_DIM, device=device))
                x_eval_img = vae.decode(z_eval).cpu()
                snapshots.append((it, x_eval_img.clone()))
            print(f"[{method}] Iter {it}/{num_iters} | loss={loss.item():.4f} | ||V||^2={v_norm:.5f}")

    return {
        'losses': losses,
        'v_norms': v_norms,
        'snapshots': snapshots,
        'method': method,
    }


In [16]:
# Run Training for all methods
results_mnist = []
for m in METHODS:
    print(f"\n{'='*60}\nTraining MNIST: {m}\n{'='*60}")
    results_mnist.append(train_mnist(method=m, num_iters=NUM_ITERS, batch_size=BATCH_SIZE, 
                                     lr_gen=LR_GEN, inner_steps=INNER_STEPS, device=DEVICE))


Training MNIST: kernel
[kernel] Warming up VAE for 400 steps...
[kernel] VAE warmup done.
[kernel] Iter 0/3000 | loss=0.2890 | ||V||^2=0.28399
[kernel] Iter 375/3000 | loss=0.3096 | ||V||^2=0.30011
[kernel] Iter 750/3000 | loss=0.2629 | ||V||^2=0.25312
[kernel] Iter 1125/3000 | loss=0.2924 | ||V||^2=0.28279
[kernel] Iter 1500/3000 | loss=0.2269 | ||V||^2=0.21765
[kernel] Iter 1875/3000 | loss=0.2268 | ||V||^2=0.21607
[kernel] Iter 2250/3000 | loss=0.2782 | ||V||^2=0.26850
[kernel] Iter 2625/3000 | loss=0.2695 | ||V||^2=0.25952
[kernel] Iter 2999/3000 | loss=0.2769 | ||V||^2=0.26670

Training MNIST: ot_direct
[ot_direct] Warming up VAE for 400 steps...
[ot_direct] VAE warmup done.
[ot_direct] Iter 0/3000 | loss=0.9660 | ||V||^2=0.96095
[ot_direct] Iter 375/3000 | loss=0.4985 | ||V||^2=0.49326
[ot_direct] Iter 750/3000 | loss=0.5528 | ||V||^2=0.54722
[ot_direct] Iter 1125/3000 | loss=0.4482 | ||V||^2=0.44270
[ot_direct] Iter 1500/3000 | loss=0.4175 | ||V||^2=0.41158
[ot_direct] Iter 187

### Visualization

In [10]:
def plot_mnist_results(results_list, save_path=SAVE_PATH):
    n_methods = len(results_list)
    n_snaps = len(results_list[0]['snapshots'])

    fig, axes = plt.subplots(n_methods, n_snaps, figsize=(3.5 * n_snaps, 3.5 * n_methods))
    if n_methods == 1:
        axes = np.array([axes])
    if n_snaps == 1:
        axes = axes.reshape(n_methods, 1)

    for row, res in enumerate(results_list):
        for col, (it, imgs) in enumerate(res['snapshots']):
            ax = axes[row, col]
            grid = make_grid(imgs[:16], nrow=4, normalize=True, value_range=(0, 1))
            grid_np = grid.permute(1, 2, 0).squeeze(-1).numpy()
            ax.imshow(grid_np, cmap='gray')
            ax.set_title(f"{res['method']} | iter {it}")
            ax.axis('off')

    plt.tight_layout()
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"Saved MNIST results to {save_path}")
    plt.show()
    plt.close(fig)

if 'results_mnist' in globals() and len(results_mnist) > 0:
    plot_mnist_results(results_mnist)
else:
    print("Run the training cell first to populate `results_mnist`.")

Saved MNIST results to results/mnist/mnist_drifting_results.png


/tmp/ipykernel_38996/3899198153.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
